# FloodWatch KP: Data Readiness & Historical Provenance Inspection
This notebook validates the meteorological ingest pipeline, verifies Open-Meteo ERA5 reanalysis coverage, checks disk caching with bounded exponential retry, and audits official historical flood records for Pakistan (2010 Super Floods & 2022 Monsoon Floods).

### Scientific Invariants & Caveats:
1. **Gridded Reanalysis vs. River Gauge:** Open-Meteo ERA5 is gridded atmospheric reanalysis (~9-30 km), NOT an in-situ local river stage or discharge gauge.
2. **Approximate Control Points:** Monitored basins are outlet points, not catchment polygons.
3. **No Synthetic Fallbacks:** If the API fails or is rate-limited (429), bounded backoff retries the request; missing coverage is reported and never fabricated.

In [ ]:
# Colab environment setup
!pip install -q pandas numpy pyyaml pyarrow requests

import sys
from pathlib import Path

# Add project root to sys.path
repo_root = Path.cwd()
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

print(f"Working directory: {repo_root}")

In [ ]:
from floodwatch.features import load_basins
import pandas as pd

basins = load_basins()
df_basins = pd.DataFrame(basins)
print(f"Loaded {len(basins)} monitored basin control points in Northern KP:")
display(df_basins[["basin_id", "name", "district", "lat", "lon", "elevation_mean_m", "primary_mechanism"]])

In [ ]:
# Test Open-Meteo ERA5 Archive Ingest with Retry and Disk Caching
from floodwatch.data_sources import openmeteo, cache
from datetime import datetime

print("Testing Open-Meteo Archive API with bounded backoff and caching...")
sample_basin = basins[0]
res = openmeteo.fetch_archive(
    basin_id=sample_basin["basin_id"],
    latitude=sample_basin["lat"],
    longitude=sample_basin["lon"],
    start_date="2022-08-25",
    end_date="2022-08-27",
    elevation_m=sample_basin.get("elevation_mean_m"),
    terrain_susceptibility=sample_basin.get("terrain_susceptibility"),
    snow_fraction=sample_basin.get("snow_fraction"),
    use_cache=True,
)

print(f"Fetch success: {len(res.frame)} hourly records retrieved.")
print(f"Source: {res.source} | Coordinates: {res.latitude}, {res.longitude} | Elevation: {res.elevation}m")
display(res.frame.head(5))

In [ ]:
# Inspect Official Historical Flood Ground Truth (2010 & 2022)
from floodwatch.data_sources import historical_events

cases = historical_events.load_historical_cases()
print(f"Loaded {len(cases)} validated historical benchmark cases:\n")

for year, case in cases.items():
    print(f"=== {year}: {case.title} ===")
    print(f"Mechanism: {case.primary_mechanism} | Window: {case.start_utc} to {case.end_utc} (Peak: {case.peak_utc})")
    print(f"Description: {case.description}\n")
    
    rows = []
    for b_id, gt in case.ground_truth.items():
        rows.append({
            "Basin ID": b_id,
            "Confirmed Status": gt.confirmed_status,
            "Spatial Granularity": gt.spatial_granularity,
            "Peak Discharge (cusecs)": gt.gauge_discharge_cusecs or "N/A",
            "Citation": gt.source_citation[:45] + "..." if len(gt.source_citation) > 45 else gt.source_citation,
        })
    display(pd.DataFrame(rows))
    print()

In [ ]:
# Provenance Citations & Reference URLs
citations = historical_events.list_citations()
print("Official Reference Citations & Provenance:")
for c in citations:
    print(f"- {c['name']} ({c['date']})")
    print(f"  Granularity: {c['granularity']} | Attribution: {c['attribution']}")
    print(f"  URL: {c['url']}\n")